# Hướng dẫn fine-tune XLM-RoBERTa cho UIT-ViQuAD 2.0

Notebook này fine-tune `xlm-roberta-base` trên `data/processed/train.json` và đánh giá bằng `data/processed/dev.json`. Phần thuật toán nằm trong `src/train_qa.py`, có giải thích tiếng Việt trong docstring và chú thích ở từng bước.

> Chạy từ từ: đọc phần lý thuyết trước, chạy từng cell theo thứ tự, và chỉ chạy cell huấn luyện khi GPU CUDA đã sẵn sàng. Một kiến trúc với ba seed sẽ chạy ba lượt, mỗi lượt ba epoch.


## 1. Bài toán đang học

Mỗi mẫu gồm câu hỏi `question`, đoạn văn `context` và đáp án chuẩn. Mô hình không viết lại câu trả lời: nó dự đoán hai vị trí token, vị trí bắt đầu (`start`) và vị trí kết thúc (`end`). Ta dùng tokenizer đã tiền huấn luyện của kiến trúc đã chọn để biến câu hỏi và đoạn văn thành token.

### Vì sao cần cửa sổ trượt?

Transformer chỉ nhận một số token giới hạn. Với `max_seq_length=384`, đoạn dài được chia thành các cửa sổ. `doc_stride=128` tạo phần chồng lấn giữa cửa sổ kế tiếp, nhờ đó đáp án gần ranh giới có cơ hội xuất hiện trọn vẹn trong ít nhất một cửa sổ. Mỗi cửa sổ trở thành một feature để mô hình học.

### Vì sao phải đổi ký tự thành token?

`answers.answer_start` là vị trí ký tự trong câu văn gốc, còn mô hình học chỉ số token. `offset_mapping` cho biết mỗi token ứng với đoạn ký tự nào; pipeline dùng nó để đổi đáp án thành cặp nhãn `start_positions` và `end_positions`. Khi dự đoán, ta dùng offsets để cắt câu trả lời nguyên văn từ context.

### Mẫu không có đáp án

Nếu `is_impossible=True`, nhãn start/end trỏ về token CLS. Khi giải mã, điểm CLS được so với span tốt nhất; nếu CLS thắng theo ngưỡng cố định 0, dự đoán là chuỗi rỗng. Ngưỡng được giữ như nhau cho các kiến trúc và seed để so sánh công bằng.


## 2. Vì sao chạy ba seed?

Seed cố định các yếu tố ngẫu nhiên chính như khởi tạo đầu phân loại và thứ tự minibatch. Một seed không thể được biết trước là tốt nhất chỉ nhờ GPU nhanh hơn. RTX 5070 Ti làm mỗi lượt chạy nhanh hơn; nó không quyết định seed nào có F1 cao nhất.

Theo quy ước trong `AGENT.md`, ta chạy `42`, `123` và `1024`, rồi xem **F1 cao nhất trên dev** để tìm seed tốt nhất đã quan sát được. Đồng thời cần báo cáo trung bình và độ lệch chuẩn qua ba seed để kết quả không phụ thuộc vào một lần chạy may mắn. GPU/CUDA vẫn có thể có một số phép tính không hoàn toàn tái lập bit-for-bit.

Cấu hình ban đầu dùng batch vật lý 4 và tích lũy gradient 8 lần, tương đương batch hiệu dụng 32. `bf16` được bật nếu GPU hỗ trợ. Nếu hết VRAM, hãy giảm batch vật lý xuống 2 và tăng `gradient_accumulation_steps` lên 16 trong `src/train_qa.py`; batch hiệu dụng vẫn là 32.


In [ ]:
from pathlib import Path  # Dùng để tìm thư mục dự án mà không ghi đường dẫn cố định.
import sys  # Dùng để Python tìm thấy các module trong thư mục src/.

project_root = Path.cwd()  # Lấy thư mục hiện tại của kernel notebook.
if project_root.name == 'notebooks':  # Nếu kernel đang ở notebooks/, lùi lên một cấp.
    project_root = project_root.parent  # Thư mục cha chính là gốc repository.
sys.path.insert(0, str(project_root))  # Thêm gốc dự án vào danh sách nơi Python tìm module.
print('Thư mục dự án:', project_root)  # In ra để người học xác nhận đúng workspace.


## 3. Kiểm tra dữ liệu và GPU trước khi chạy

Hai JSONL cần được tạo từ notebook EDA. Cần dùng môi trường Python đã cài các thư viện trong `requirements.txt`, đặc biệt là bản PyTorch có CUDA. Nếu không thấy GPU ở cell dưới, đừng chạy fine-tune vì CPU sẽ chậm hơn rất nhiều.


In [ ]:
import torch  # PyTorch kiểm tra GPU và thực hiện phép tính của mô hình.
from src.train_qa import TRAIN_FILE, DEV_FILE  # Lấy đường dẫn dữ liệu mà script sẽ dùng.

print('Train tồn tại:', TRAIN_FILE.exists(), TRAIN_FILE)  # Xác nhận tệp train đã sẵn sàng.
print('Dev tồn tại:', DEV_FILE.exists(), DEV_FILE)  # Xác nhận tệp dev đã sẵn sàng.
print('Có CUDA:', torch.cuda.is_available())  # CUDA cho phép PyTorch dùng GPU NVIDIA.
if torch.cuda.is_available():  # Chỉ hỏi tên thiết bị khi đã phát hiện GPU.
    print('GPU:', torch.cuda.get_device_name(0))  # Kiểm tra notebook nhận đúng RTX 5070 Ti.
    print('Hỗ trợ BF16:', torch.cuda.is_bf16_supported())  # BF16 thường tiết kiệm bộ nhớ và ổn định.


## 4. Chạy thí nghiệm

Cell sau gọi cùng một pipeline dùng cho hai kiến trúc. Với mỗi seed, script tạo mô hình mới từ checkpoint gốc, token hóa dữ liệu, fine-tune ba epoch, đánh giá EM/F1 sau từng epoch và lưu checkpoint tốt nhất theo F1. Không tái sử dụng trọng số của seed trước.

Mỗi lượt sẽ ghi `metrics.json`, `predictions.json` và thư mục `best_model/` dưới `results/finetune/`. Xem đầy đủ logic từng dòng trong `src/train_qa.py`; đặc biệt là `prepare_train_features`, `decode_predictions` và `make_compute_metrics`.


In [ ]:
from src.train_qa import run_experiment  # Nạp hàm điều phối tiền xử lý, fine-tune và đánh giá.

seeds = [42, 123, 1024]  # Ba seed theo quy ước của project để đo độ biến động.
model_names = ['xlm-roberta']  # Notebook này chỉ chạy kiến trúc xlm-roberta-base.
run_experiment(model_names, seeds)  # Chạy lần lượt từng seed và lưu kết quả vào results/.


## 5. Đọc kết quả

`summary.json` ghi seed có F1 cao nhất và điểm trung bình ± độ lệch chuẩn của các seed trong lần chạy. `predictions.json` lưu câu trả lời từng mẫu để xem lỗi. Tên `best_seed` chỉ có nghĩa là seed tốt nhất trong ba seed đã chạy trên tập dev này; nó không đảm bảo sẽ tốt nhất trên tập test hoặc dữ liệu mới.

Khi so sánh XLM-RoBERTa với mDeBERTa, so sánh cả F1/EM trung bình và độ lệch chuẩn, dùng cùng train/dev, seed, epoch, độ dài token và ngưỡng no-answer. Sau khi chọn cấu hình, chỉ dùng test set một lần để báo cáo cuối cùng nếu dự án có test set riêng.


In [ ]:
import json  # Đọc tệp kết quả JSON do script tạo ra.
from pathlib import Path  # Tạo đường dẫn kết quả tương đối với gốc repository.

summary_path = project_root / 'results' / 'finetune' / 'xlm-roberta' / 'summary.json'  # Tổng hợp riêng model này.
if summary_path.exists():  # Chỉ đọc sau khi cell huấn luyện đã chạy xong.
    print(summary_path.read_text(encoding='utf-8'))  # Hiển thị seed tốt nhất, trung bình và độ lệch chuẩn.
else:  # Thí nghiệm có thể chưa chạy hoặc chưa kết thúc.
    print('Chưa có summary; hãy chạy cell huấn luyện đến khi hoàn tất.')  # Thông báo bước kế tiếp.
